# Diabetes Risk Prediction
## Model Training & Evaluation
This notebook contains all model training experiments across 4 models and 4 balancing strategies (16 total combinations).

## Load the Unscaled Training Data

In [13]:
# Load preprocessed data
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression

X_train = np.load('data/X_train.npy')
X_test = np.load('data/X_test.npy')
y_train = np.load('data/y_train.npy')
y_test = np.load('data/y_test.npy')
feature_names = pd.read_csv('data/feature_names.csv').values.flatten()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Feature names:", feature_names)

X_train: (202944, 21)
y_train: (202944,)
Feature names: ['HighBP' 'HighChol' 'CholCheck' 'BMI' 'Smoker' 'Stroke'
 'HeartDiseaseorAttack' 'PhysActivity' 'Fruits' 'Veggies'
 'HvyAlcoholConsump' 'AnyHealthcare' 'NoDocbcCost' 'GenHlth' 'MentHlth'
 'PhysHlth' 'DiffWalk' 'Sex' 'Age' 'Education' 'Income']


## Training: Raw Data (No Balancing)

We start with the raw unbalanced data as our baseline.

### Lets Load the Scaled Data

In [14]:
X_train_scaled = np.load('data/X_train_scaled.npy')
X_test_scaled = np.load('data/X_test_scaled.npy')

print("Scaled training set:", X_train_scaled.shape)

Scaled training set: (202944, 21)


### Logistic Regression on Raw Training Data

In [15]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_predictions = lr_model.predict(X_test_scaled)

print("Logistic Regression - Raw Data Results:\n")
print(classification_report(y_test, lr_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, lr_model.predict_proba(X_test_scaled)[:,1]), 4))

Logistic Regression - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.97      0.92     42741
    Diabetes       0.55      0.19      0.28      7995

    accuracy                           0.85     50736
   macro avg       0.71      0.58      0.60     50736
weighted avg       0.82      0.85      0.82     50736

AUC-ROC: 0.8172


#### Logistic Regression - Raw Data Interpretation

Logistic Regression on raw data shows the same imbalance problem as RF and XGBoost. Recall is only 0.19, meaning it catches even fewer diabetic patients than RF and XGBoost (both at 0.21). However, its AUC-ROC (0.8172) is comparable to XGBoost (0.8194), meaning it ranks patients almost as well internally despite being a much simpler model. This suggests the relationship between features and diabetes risk is fairly linear in this dataset, which is a good sign for LR as a baseline.

### KNN on Raw Training Data

In [16]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_predictions = knn_model.predict(X_test_scaled)

print("KNN - Raw Data Results:\n")
print(classification_report(y_test, knn_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, knn_model.predict_proba(X_test_scaled)[:,1]), 4))

KNN - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.94      0.90     42741
    Diabetes       0.43      0.24      0.31      7995

    accuracy                           0.83     50736
   macro avg       0.65      0.59      0.61     50736
weighted avg       0.80      0.83      0.81     50736

AUC-ROC: 0.7259


#### KNN - Raw Data Interpretation

KNN has the lowest AUC-ROC of all four models (0.73 vs 0.79-0.82 for the others), meaning it's the worst at ranking patients overall. Its recall (0.24) is slightly better than LR (0.19) but still misses 76% of diabetic patients. KNN's weakness here likely comes from the high dimensionality (21 features) and the imbalanced data. With so many more healthy patients, most of any patient's 5 nearest neighbors tend to be healthy, biasing predictions toward "no diabetes."

### Random Forest on Raw Training Data

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)

# Results
print("Random Forest - Raw Data Results:\n")
print(classification_report(y_test, rf_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]), 4))

Random Forest - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.96      0.91     42741
    Diabetes       0.52      0.21      0.30      7995

    accuracy                           0.84     50736
   macro avg       0.69      0.59      0.61     50736
weighted avg       0.81      0.84      0.82     50736

AUC-ROC: 0.7921


#### Random Forest - Raw Data Interpretation

The results confirm the class imbalance problem we identified during EDA. The model achieves 84% accuracy, but this is misleading because it's mostly just correctly predicting healthy patients (96% recall) while only catching 21% of actual diabetic patients.

Key takeaways:
- The model misses roughly 4 out of every 5 diabetic patients (recall = 0.21)
- When it does flag someone as diabetic, it's only right about half the time (precision = 0.52)
- The F1-score for diabetes is 0.30, which is poor
- AUC-ROC of 0.79 shows the model has some ability to distinguish between classes but struggles significantly with the minority class

This baseline result demonstrates exactly why balancing techniques like SMOTE and SMOTE-ENN are needed. Without addressing the imbalance, the model effectively learns to default to predicting "no diabetes" because that's the safe bet given the 84/16 class split.

### XGBoost on Raw Training Data
Training XGBoost with default hyperparameters on the raw unbalanced dataset.
XGBoost builds trees sequentially, where each new tree focuses on correcting the mistakes of the previous ones.

In [18]:
from xgboost import XGBClassifier

# Train XGBoost
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_predictions = xgb_model.predict(X_test)

# Results
print("XGBoost - Raw Data Results:\n")
print(classification_report(y_test, xgb_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]), 4))

XGBoost - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.97      0.92     42741
    Diabetes       0.57      0.21      0.30      7995

    accuracy                           0.85     50736
   macro avg       0.72      0.59      0.61     50736
weighted avg       0.82      0.85      0.82     50736

AUC-ROC: 0.8194


#### XGBoost - Raw Data Interpretation

XGBoost performs very similarly to Random Forest on the raw data. Diabetes recall is identical at 0.21 and F1-score is the same at 0.30. XGBoost shows slight improvement in precision (0.57 vs 0.52) and AUC-ROC (0.82 vs 0.79), suggesting it ranks patients slightly better overall.

The key finding here is that switching to a more powerful model alone does not solve the class imbalance problem. Both models default to predicting "no diabetes" for the vast majority of cases. This confirms that the issue lies in the data distribution, not the model architecture, and motivates the need for resampling techniques like SMOTE and SMOTE-ENN.

## Balancing Techniques
The raw data has an 84/16 split (healthy vs diabetic). We apply three techniques to the training data only (never the test data) to address this imbalance.

The raw training data has a significant class imbalance: 170,962 healthy (class 0) vs 31,982 diabetic (class 1). We apply three balancing techniques to the training data only. The test data is never touched so it continues to represent real-world conditions.

### How each technique works:

**SMOTE (Synthetic Minority Oversampling Technique):** Creates new synthetic diabetic patients by finding pairs of real diabetic patients that are close to each other and generating a new data point somewhere in between them. This is repeated until both classes are equal in size.

**SMOTE-ENN (SMOTE + Edited Nearest Neighbors):** First applies SMOTE to create synthetic diabetic patients. Then runs ENN, which removes noisy or confusing data points from both classes. A point is removed if most of its nearest neighbors belong to a different class. This produces cleaner data but the two classes may not be perfectly equal.

**Random Undersampling:** Randomly deletes healthy patients until both classes are equal. Simple and fast but discards a large portion of the training data.

### Balancing of Raw Data

In [19]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler

# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("SMOTE:", np.unique(y_train_smote, return_counts=True))

# SMOTE-ENN
smote_enn = SMOTEENN(random_state=42)
X_train_smoteenn, y_train_smoteenn = smote_enn.fit_resample(X_train, y_train)
print("SMOTE-ENN:", np.unique(y_train_smoteenn, return_counts=True))

# Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
print("Undersampling:", np.unique(y_train_under, return_counts=True))

print("\nOriginal:", np.unique(y_train, return_counts=True))

SMOTE: (array([0, 1]), array([170962, 170962]))
SMOTE-ENN: (array([0, 1]), array([ 98808, 159903]))
Undersampling: (array([0, 1]), array([31982, 31982]))

Original: (array([0, 1]), array([170962,  31982]))


#### Balancing Results:

| Technique | Healthy (0) | Diabetic (1) | Total Samples |
|---|---|---|---|
| Original | 170,962 | 31,982 | 202,944 |
| SMOTE | 170,962 | 170,962 | 341,924 |
| SMOTE-ENN | 98,808 | 159,903 | 258,711 |
| Undersampling | 31,982 | 31,982 | 63,964 |

SMOTE doubled the dataset by generating ~139,000 synthetic diabetic patients. SMOTE-ENN also generated synthetic patients but then cleaned up noisy points from both classes, which is why the healthy count dropped from 170,962 to 98,808. The classes are not perfectly equal but the data is cleaner overall. Undersampling achieved perfect balance but at a steep cost, reducing the total dataset from 202,944 to only 63,964 samples (a 68% reduction in data).

### Balancing on Scaled Data (for LR and KNN)
LR and KNN require scaled features. We apply the same three balancing techniques to the scaled training data.

In [20]:
X_train_smote_scaled, y_train_smote_scaled = SMOTE(random_state=42).fit_resample(X_train_scaled, y_train)
X_train_smoteenn_scaled, y_train_smoteenn_scaled = SMOTEENN(random_state=42).fit_resample(X_train_scaled, y_train)
X_train_under_scaled, y_train_under_scaled = RandomUnderSampler(random_state=42).fit_resample(X_train_scaled, y_train)

print("SMOTE (scaled):", np.unique(y_train_smote_scaled, return_counts=True))
print("SMOTE-ENN (scaled):", np.unique(y_train_smoteenn_scaled, return_counts=True))
print("Undersampling (scaled):", np.unique(y_train_under_scaled, return_counts=True))

SMOTE (scaled): (array([0, 1]), array([170962, 170962]))
SMOTE-ENN (scaled): (array([0, 1]), array([109963, 148049]))
Undersampling (scaled): (array([0, 1]), array([31982, 31982]))


#### Balancing Results (Scaled Data - for LR and KNN):

| Technique | Healthy (0) | Diabetic (1) | Total Samples |
|---|---|---|---|
| Original | 170,962 | 31,982 | 202,944 |
| SMOTE | 170,962 | 170,962 | 341,924 |
| SMOTE-ENN | 109,963 | 148,049 | 258,012 |
| Undersampling | 31,982 | 31,982 | 63,964 |

Note: SMOTE-ENN produces different counts on scaled vs unscaled data (98,808/159,903 vs 109,963/148,049) because the ENN cleaning step uses nearest neighbor distances, which are affected by scaling. SMOTE and undersampling counts are identical across both versions.

## SMOTE - Model Training


### Logistic Regression - SMOTE

In [ ]:
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote_scaled, y_train_smote_scaled)
lr_smote_pred = lr_smote.predict(X_test_scaled)

print("Logistic Regression - SMOTE Results:\n")
print(classification_report(y_test, lr_smote_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, lr_smote.predict_proba(X_test_scaled)[:,1]), 4))

Logistic Regression - SMOTE Results:

              precision    recall  f1-score   support

 No Diabetes       0.94      0.73      0.82     42741
    Diabetes       0.34      0.76      0.47      7995

    accuracy                           0.73     50736
   macro avg       0.64      0.74      0.64     50736
weighted avg       0.85      0.73      0.76     50736

AUC-ROC: 0.8165


### KNN - SMOTE

In [ ]:
knn_smote = KNeighborsClassifier(n_neighbors=5)
knn_smote.fit(X_train_smote_scaled, y_train_smote_scaled)
knn_smote_pred = knn_smote.predict(X_test_scaled)

print("KNN - SMOTE Results:\n")
print(classification_report(y_test, knn_smote_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, knn_smote.predict_proba(X_test_scaled)[:,1]), 4))

KNN - SMOTE Results:

              precision    recall  f1-score   support

 No Diabetes       0.91      0.73      0.81     42741
    Diabetes       0.30      0.63      0.41      7995

    accuracy                           0.71     50736
   macro avg       0.61      0.68      0.61     50736
weighted avg       0.82      0.71      0.75     50736

AUC-ROC: 0.7253


### Random Forest - SMOTE

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Random Forest on SMOTE data
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)
rf_smote_pred = rf_smote.predict(X_test)

print("Random Forest - SMOTE Results:\n")
print(classification_report(y_test, rf_smote_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_smote.predict_proba(X_test)[:,1]), 4))

Random Forest - SMOTE Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.95      0.91     42741
    Diabetes       0.49      0.24      0.32      7995

    accuracy                           0.84     50736
   macro avg       0.68      0.60      0.62     50736
weighted avg       0.81      0.84      0.82     50736

AUC-ROC: 0.7905


### XGBoost - SMOTE

In [ ]:
# XGBoost on SMOTE data
xgb_smote = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_smote.fit(X_train_smote, y_train_smote)
xgb_smote_pred = xgb_smote.predict(X_test)

print("XGBoost - SMOTE Results:\n")
print(classification_report(y_test, xgb_smote_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_smote.predict_proba(X_test)[:,1]), 4))

XGBoost - SMOTE Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.97      0.92     42741
    Diabetes       0.57      0.23      0.33      7995

    accuracy                           0.85     50736
   macro avg       0.72      0.60      0.62     50736
weighted avg       0.82      0.85      0.82     50736

AUC-ROC: 0.8205


## SMOTE-ENN - Model Training

### Logistic Regression - SMOTE-ENN

In [33]:
lr_smoteenn = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_smoteenn.fit(X_train_smoteenn_scaled, y_train_smoteenn_scaled)
lr_smoteenn_pred = lr_smoteenn.predict(X_test_scaled)

print("Logistic Regression - SMOTE-ENN Results:\n")
print(classification_report(y_test, lr_smoteenn_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, lr_smoteenn.predict_proba(X_test_scaled)[:,1]), 4))

Logistic Regression - SMOTE-ENN Results:

              precision    recall  f1-score   support

 No Diabetes       0.96      0.64      0.77     42741
    Diabetes       0.30      0.84      0.45      7995

    accuracy                           0.67     50736
   macro avg       0.63      0.74      0.61     50736
weighted avg       0.85      0.67      0.71     50736

AUC-ROC: 0.8176


### KNN - SMOTE-ENN

In [22]:
knn_smoteenn = KNeighborsClassifier(n_neighbors=5)
knn_smoteenn.fit(X_train_smoteenn_scaled, y_train_smoteenn_scaled)
knn_smoteenn_pred = knn_smoteenn.predict(X_test_scaled)

print("KNN - SMOTE-ENN Results:\n")
print(classification_report(y_test, knn_smoteenn_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, knn_smoteenn.predict_proba(X_test_scaled)[:,1]), 4))

KNN - SMOTE-ENN Results:

              precision    recall  f1-score   support

 No Diabetes       0.94      0.64      0.76     42741
    Diabetes       0.29      0.77      0.42      7995

    accuracy                           0.66     50736
   macro avg       0.61      0.71      0.59     50736
weighted avg       0.83      0.66      0.71     50736

AUC-ROC: 0.7439


### Random Forest - SMOTE-ENN

In [23]:
# Random Forest on SMOTE-ENN data
rf_smoteenn = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smoteenn.fit(X_train_smoteenn, y_train_smoteenn)
rf_smoteenn_pred = rf_smoteenn.predict(X_test)

print("Random Forest - SMOTE-ENN Results:\n")
print(classification_report(y_test, rf_smoteenn_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_smoteenn.predict_proba(X_test)[:,1]), 4))

Random Forest - SMOTE-ENN Results:

              precision    recall  f1-score   support

 No Diabetes       0.93      0.78      0.85     42741
    Diabetes       0.37      0.68      0.48      7995

    accuracy                           0.77     50736
   macro avg       0.65      0.73      0.66     50736
weighted avg       0.84      0.77      0.79     50736

AUC-ROC: 0.8072


### XGBoost - SMOTE-ENN

In [24]:
# XGBoost on SMOTE-ENN data
xgb_smoteenn = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_smoteenn.fit(X_train_smoteenn, y_train_smoteenn)
xgb_smoteenn_pred = xgb_smoteenn.predict(X_test)

print("XGBoost - SMOTE-ENN Results:\n")
print(classification_report(y_test, xgb_smoteenn_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_smoteenn.predict_proba(X_test)[:,1]), 4))

XGBoost - SMOTE-ENN Results:

              precision    recall  f1-score   support

 No Diabetes       0.93      0.79      0.85     42741
    Diabetes       0.38      0.68      0.48      7995

    accuracy                           0.77     50736
   macro avg       0.65      0.73      0.67     50736
weighted avg       0.84      0.77      0.80     50736

AUC-ROC: 0.8198


## Undersampling - Model Training

### Logistic Regression - Undersampling

In [ ]:
lr_under = LogisticRegression(max_iter=1000, random_state=42)
lr_under.fit(X_train_under_scaled, y_train_under_scaled)
lr_under_pred = lr_under.predict(X_test_scaled)

print("Logistic Regression - Undersampling Results:\n")
print(classification_report(y_test, lr_under_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, lr_under.predict_proba(X_test_scaled)[:,1]), 4))

Logistic Regression - Undersampling Results:

              precision    recall  f1-score   support

 No Diabetes       0.94      0.73      0.82     42741
    Diabetes       0.34      0.76      0.47      7995

    accuracy                           0.73     50736
   macro avg       0.64      0.74      0.64     50736
weighted avg       0.85      0.73      0.76     50736

AUC-ROC: 0.8177


### KNN - Undersampling

In [ ]:
knn_under = KNeighborsClassifier(n_neighbors=5)
knn_under.fit(X_train_under_scaled, y_train_under_scaled)
knn_under_pred = knn_under.predict(X_test_scaled)

print("KNN - Undersampling Results:\n")
print(classification_report(y_test, knn_under_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, knn_under.predict_proba(X_test_scaled)[:,1]), 4))

KNN - Undersampling Results:

              precision    recall  f1-score   support

 No Diabetes       0.93      0.68      0.79     42741
    Diabetes       0.30      0.73      0.43      7995

    accuracy                           0.69     50736
   macro avg       0.62      0.71      0.61     50736
weighted avg       0.83      0.69      0.73     50736

AUC-ROC: 0.7628


### Random Forest - Undersampling

In [ ]:
# Random Forest on Undersampled data
rf_under = RandomForestClassifier(n_estimators=100, random_state=42)
rf_under.fit(X_train_under, y_train_under)
rf_under_pred = rf_under.predict(X_test)

print("Random Forest - Undersampling Results:\n")
print(classification_report(y_test, rf_under_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_under.predict_proba(X_test)[:,1]), 4))

Random Forest - Undersampling Results:

              precision    recall  f1-score   support

 No Diabetes       0.94      0.70      0.80     42741
    Diabetes       0.32      0.76      0.45      7995

    accuracy                           0.71     50736
   macro avg       0.63      0.73      0.63     50736
weighted avg       0.84      0.71      0.75     50736

AUC-ROC: 0.7997


### XGBoost - Undersampling

In [ ]:
# XGBoost on Undersampled data
xgb_under = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_under.fit(X_train_under, y_train_under)
xgb_under_pred = xgb_under.predict(X_test)

print("XGBoost - Undersampling Results:\n")
print(classification_report(y_test, xgb_under_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_under.predict_proba(X_test)[:,1]), 4))

XGBoost - Undersampling Results:

              precision    recall  f1-score   support

 No Diabetes       0.94      0.70      0.81     42741
    Diabetes       0.33      0.78      0.46      7995

    accuracy                           0.72     50736
   macro avg       0.64      0.74      0.63     50736
weighted avg       0.85      0.72      0.75     50736

AUC-ROC: 0.8158


## Results Summary

In [ ]:
all_results = pd.DataFrame({
    'Model': ['Random Forest']*4 + ['XGBoost']*4 + ['Logistic Regression']*4 + ['KNN']*4,
    'Balancing': ['Raw', 'SMOTE', 'SMOTE-ENN', 'Undersampling']*4,
    'Diabetes Precision': [0.52, 0.49, 0.37, 0.32, 0.57, 0.57, 0.38, 0.33, 0.55, 0.34, 0.30, 0.34, 0.43, 0.30, 0.29, 0.30],
    'Diabetes Recall': [0.21, 0.24, 0.68, 0.76, 0.21, 0.23, 0.68, 0.78, 0.19, 0.76, 0.84, 0.76, 0.24, 0.63, 0.77, 0.73],
    'Diabetes F1': [0.30, 0.32, 0.48, 0.45, 0.30, 0.33, 0.48, 0.46, 0.28, 0.47, 0.45, 0.47, 0.31, 0.41, 0.42, 0.43],
    'AUC-ROC': [0.7921, 0.7905, 0.8072, 0.7997, 0.8194, 0.8205, 0.8198, 0.8158, 0.8172, 0.8165, 0.8176, 0.8177, 0.7259, 0.7253, 0.7439, 0.7628]
})

print(all_results.to_string(index=False))

              Model     Balancing  Diabetes Precision  Diabetes Recall  Diabetes F1  AUC-ROC
      Random Forest           Raw                0.52             0.21         0.30   0.7921
      Random Forest         SMOTE                0.49             0.24         0.32   0.7905
      Random Forest     SMOTE-ENN                0.37             0.68         0.48   0.8072
      Random Forest Undersampling                0.32             0.76         0.45   0.7997
            XGBoost           Raw                0.57             0.21         0.30   0.8194
            XGBoost         SMOTE                0.57             0.23         0.33   0.8205
            XGBoost     SMOTE-ENN                0.38             0.68         0.48   0.8198
            XGBoost Undersampling                0.33             0.78         0.46   0.8158
Logistic Regression           Raw                0.55             0.19         0.28   0.8172
Logistic Regression         SMOTE                0.34             0.76

### Interpretation of Results

**Key findings from 8 experiments (Bilal's models):**

**1. Plain SMOTE barely helped recall.**
Both RF and XGBoost saw almost no improvement in diabetes recall with SMOTE (0.21 to 0.24 for RF, 0.21 to 0.23 for XGBoost). Interestingly, XGBoost + SMOTE achieved the highest AUC-ROC across all experiments (0.8205), meaning the model got slightly better at ranking patients internally, but this did not translate into actually catching more diabetic patients.

**2. SMOTE-ENN made the biggest practical difference.**
Diabetes recall jumped from 0.21 to 0.68 for both models. The F1-score improved from 0.30 to 0.48 for both. The cleaning step (ENN) appears to be critical. By removing noisy data points near the class boundary, the model gets a clearer picture of what separates healthy from diabetic patients.

**3. Undersampling gave the highest recall but lowest precision.**
Diabetes recall reached 0.76 (RF) and 0.78 (XGBoost), the highest across all experiments. However, precision dropped to 0.32 (RF) and 0.33 (XGBoost), meaning roughly 2 out of every 3 diabetes flags were false alarms. F1-scores (0.45 and 0.46) were also slightly lower than SMOTE-ENN.

**4. XGBoost consistently had higher AUC-ROC than Random Forest.**
Across all four balancing strategies, XGBoost had a higher AUC-ROC score (range: 0.8158-0.8205) compared to Random Forest (range: 0.7905-0.8072).

**5. There is a clear precision-recall trade-off.**
As balancing techniques push recall higher (catching more diabetic patients), precision drops (more false alarms). The best technique depends on the clinical context. If missing a diabetic patient is more dangerous than a false alarm, undersampling is preferred. If false alarms are costly, SMOTE-ENN offers a better balance.

**Best combination so far: XGBoost + SMOTE-ENN** offers the strongest balance of recall (0.68), F1-score (0.48), and AUC-ROC (0.8198).

The reason I call SMOTE-ENN the "best balance" is because it's not the best at any single metric, but it's strong across all of them. It doesn't sacrifice too much precision for recall, and it doesn't sacrifice too much recall for precision. It's the most well-rounded.

### Interpretation of All 16 Experiments

**1. All models suffer equally on raw data.**
Every model trained on raw imbalanced data achieved recall between 0.19-0.24. Regardless of model complexity (from simple LR to powerful XGBoost), none could reliably detect diabetic patients without balancing. This confirms the class imbalance is the core problem, not model choice.

**2. SMOTE had mixed results across models.**
For RF and XGBoost, SMOTE barely moved recall (0.21 to 0.24 and 0.23). But for LR, SMOTE made a huge difference: recall jumped from 0.19 to 0.76. KNN also improved significantly (0.24 to 0.63). This suggests that linear and distance-based models benefit more from having additional synthetic minority samples than tree-based models do.

**3. SMOTE-ENN consistently pushed recall the highest.**
LR + SMOTE-ENN achieved the highest recall of any experiment at 0.84, catching 5 out of every 6 diabetic patients. KNN + SMOTE-ENN hit 0.77. RF and XGBoost both reached 0.68. The ENN cleaning step helps all models, but helps LR and KNN more aggressively because these models are more sensitive to noisy data points near the decision boundary.

**4. Undersampling performed similarly to SMOTE for most models.**
LR + Undersampling (recall 0.76, F1 0.47) was nearly identical to LR + SMOTE (recall 0.76, F1 0.47). XGBoost + Undersampling achieved the highest recall among tree models (0.78). KNN + Undersampling (recall 0.73) outperformed KNN + SMOTE (0.63), likely because the smaller dataset made distance calculations more meaningful.

**5. KNN was the weakest model across all conditions.**
KNN's AUC-ROC ranged from 0.7253-0.7628, well below the other three models (all above 0.79). KNN struggles with high-dimensional data and is computationally expensive on large datasets. It is not suitable for this problem at scale.

**6. LR, XGBoost, and RF are competitive with each other.**
LR, XGBoost, and RF all achieved AUC-ROC above 0.79 across all balancing strategies. The simpler LR model matched XGBoost in AUC-ROC (both ~0.82), supporting the professor's point about being grounded in fundamentals: a simple model with proper data handling can match a complex one.

**7. Best combinations depend on the use case:**

| Goal | Best Combination | Recall | F1 | AUC-ROC |
|---|---|---|---|---|
| Highest recall (catch most patients) | LR + SMOTE-ENN | 0.84 | 0.45 | 0.8176 |
| Best F1 balance | RF + SMOTE-ENN or XGBoost + SMOTE-ENN | 0.68 | 0.48 | 0.8072/0.8198 |
| Best AUC-ROC (ranking ability) | XGBoost + SMOTE | 0.23 | 0.33 | 0.8205 |
| Best recall with good F1 | LR + SMOTE or LR + Undersampling | 0.76 | 0.47 | 0.8165/0.8177 |

For a diabetes screening tool where missing patients is the biggest risk, **LR + SMOTE-ENN** is the recommended choice with 84% recall. For a balanced approach, **XGBoost + SMOTE-ENN** offers the best F1 (0.48) with strong AUC-ROC (0.8198).

## Hyperparameter Tuning
We use GridSearchCV to systematically search for the best hyperparameters on our two strongest models: XGBoost + SMOTE-ENN and LR + SMOTE-ENN. GridSearchCV tries every combination of specified hyperparameter values using 5-fold cross-validation, then reports the combination that gives the best score.

### XGBoost + SMOTE-ENN

#### XGBoost Hyperparameter tuning - SMOTE-ENN

In [25]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# XGBoost hyperparameter tuning on SMOTE-ENN data
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid_xgb,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("Starting XGBoost GridSearchCV (27 combinations x 5 folds = 135 fits)...")
xgb_grid.fit(X_train_smoteenn, y_train_smoteenn)

print(f"\nBest parameters: {xgb_grid.best_params_}")
print(f"Best F1 score (CV): {round(xgb_grid.best_score_, 4)}")

Starting XGBoost GridSearchCV (27 combinations x 5 folds = 135 fits)...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 50}
Best F1 score (CV): 0.9278


#### Testing: XGBoost - Tuned on SMOTE-ENN
Retrain with best hyperparameters and evaluate on the real imbalanced test set.

In [26]:
# Retrain with best hyperparameters on full SMOTE-ENN data
xgb_tuned = XGBClassifier(
    n_estimators=50,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_tuned.fit(X_train_smoteenn, y_train_smoteenn)
xgb_tuned_pred = xgb_tuned.predict(X_test)

print("XGBoost Tuned - SMOTE-ENN Results:\n")
print(classification_report(y_test, xgb_tuned_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_tuned.predict_proba(X_test)[:,1]), 4))

# Compare with previous untuned results
print("\nComparison with untuned XGBoost + SMOTE-ENN:")
print("  Untuned: Recall=0.68, F1=0.48, AUC-ROC=0.8198")

XGBoost Tuned - SMOTE-ENN Results:

              precision    recall  f1-score   support

 No Diabetes       0.94      0.72      0.82     42741
    Diabetes       0.34      0.76      0.47      7995

    accuracy                           0.73     50736
   macro avg       0.64      0.74      0.64     50736
weighted avg       0.85      0.73      0.76     50736

AUC-ROC: 0.82

Comparison with untuned XGBoost + SMOTE-ENN:
  Untuned: Recall=0.68, F1=0.48, AUC-ROC=0.8198


#### XGBoost Tuning Interpretation

**Best hyperparameters found:** n_estimators=50, max_depth=5, learning_rate=0.1

**What the hyperparameters tell us:**
The optimal model is actually simpler than our default (50 trees vs 100, depth 5 vs unlimited). This suggests the untuned model was slightly overcomplicating its decisions. Fewer, shallower trees produced a model that generalizes better to unseen patients rather than memorizing quirks in the training data.

**Impact on real test data:**
Tuning improved recall from 0.68 to 0.76 (catching 8% more diabetic patients) while keeping AUC-ROC stable at 0.82. F1 dropped marginally from 0.48 to 0.47 due to a slight precision decrease (0.38 to 0.34). The tuned model is more aggressive at flagging patients, which is the preferred behavior for a screening tool where missing a diabetic patient is more costly than a false alarm.

**Key takeaway:** Hyperparameter tuning shifted the model's behavior in a clinically useful direction. A simpler, well-tuned model outperformed the more complex default on the metric that matters most for healthcare: recall.

### Logistic Regression + SMOTE-ENN

#### Logistic Regression Hyperparameter Tuning - SMOTE-ENN

In [27]:
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("Starting LR GridSearchCV (4 combinations x 5 folds = 20 fits)...")
lr_grid.fit(X_train_smoteenn_scaled, y_train_smoteenn_scaled)

print(f"\nBest parameters: {lr_grid.best_params_}")
print(f"Best F1 score (CV): {round(lr_grid.best_score_, 4)}")

Starting LR GridSearchCV (4 combinations x 5 folds = 20 fits)...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'C': 0.1}
Best F1 score (CV): 0.8795


#### Testing: Logistic Regression - Tuned on SMOTE-ENN
Retrain with best hyperparameter and evaluate on the real imbalanced test set.

In [28]:
lr_tuned = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr_tuned.fit(X_train_smoteenn_scaled, y_train_smoteenn_scaled)
lr_tuned_pred = lr_tuned.predict(X_test_scaled)

print("LR Tuned - SMOTE-ENN Results:\n")
print(classification_report(y_test, lr_tuned_pred, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, lr_tuned.predict_proba(X_test_scaled)[:,1]), 4))

print("\nComparison with untuned LR + SMOTE-ENN:")
print("  Untuned: Recall=0.84, F1=0.45, AUC-ROC=0.8176")

LR Tuned - SMOTE-ENN Results:

              precision    recall  f1-score   support

 No Diabetes       0.96      0.64      0.77     42741
    Diabetes       0.30      0.84      0.45      7995

    accuracy                           0.67     50736
   macro avg       0.63      0.74      0.61     50736
weighted avg       0.85      0.67      0.71     50736

AUC-ROC: 0.8176

Comparison with untuned LR + SMOTE-ENN:
  Untuned: Recall=0.84, F1=0.45, AUC-ROC=0.8176


#### Logistic Regression Tuning Interpretation

**Best hyperparameter found:** C=0.1 (default was C=1.0)

**Impact on real test data:** None. Every metric is identical to the untuned model (Recall=0.84, F1=0.45, AUC-ROC=0.8176). Changing C by a factor of 10 (C=1.0) or even 100 (C=10) had zero effect on predictions.

**What this tells us:** Logistic Regression is highly stable on this dataset. The decision boundary it learns is robust and not sensitive to regularization strength. This is actually a strength: it means the model's behavior is reliable and reproducible regardless of this hyperparameter. Combined with its simplicity and interpretability, this further supports LR as a strong candidate for a clinical screening tool where consistency matters.

Why changing C didn't change anything:
Think about it this way. Our dataset has 21 features and 258,000 training samples after SMOTE-ENN. That's a LOT of data compared to the number of features. Regularization mainly helps when you have the opposite situation: lots of features and not much data. In that case, the model is tempted to find fake patterns because there isn't enough data to confirm what's real.
With 258,000 samples, the model has more than enough evidence to figure out the real patterns. Whether you tell it "be conservative" (C=0.1) or "be relaxed" (C=1.0), it arrives at the same answer because the data is so overwhelming that there's really only one sensible decision boundary to find.
It's like asking someone to find a path through a maze that has only one path. Whether you tell them "be careful" or "go fast," they end up at the same exit because there's only one way to go.